# Training OWN tokenizer for URDU

In [1]:
from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel

# To give vector embedding and token embedding
import torch
import torch.nn as nn



c:\Users\Axil\Desktop\ScratchLLM\Baat Urdu_l4\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Streaming the dataset to avoid loading it all into memory at once
print("Loading dataset...")
OG_urdu_dataset = load_dataset("allenai/c4", "ur",split="train", streaming=True) # 70%
Rom_urdu_dataset=load_dataset("Khubaib01/RomanUrdu-NLP-Sentiment-Corpus", split="train", streaming=True)  #15%
ENG_dataset = load_dataset("HuggingFaceFW/fineweb-edu", name="CC-MAIN-2024-10", split="train", streaming=True) #15%

Loading dataset...


c:\Users\Axil\Desktop\ScratchLLM\Baat Urdu_l4\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Axil\.cache\huggingface\hub\datasets--Khubaib01--RomanUrdu-NLP-Sentiment-Corpus. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\Axil\Desktop\ScratchLLM\Baat Urdu_l4\.venv\Lib\site-packages\huggi

In [5]:


# Intialize a BPE tokenizer
tokenizer = Tokenizer(BPE(unk_token="[unk]"))
tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)

#Configure the tokenizer trainer
trainer=BpeTrainer(vocab_size=32000,special_tokens=["[unk]","[pad]","[endoftext]"],show_progress=True)



In [8]:
#Feeding text to trainer in batches
# OG urdu
def batch_iterator(batch_size=1000,max_docs=500000):

    urdu_iter=iter(OG_urdu_dataset)
    rom_iter=iter(Rom_urdu_dataset)
    eng_iter=iter(ENG_dataset)

    batch = []
    count = 0

    while count<max_docs:
        for _ in range(70):
            try:
                batch.append(next(urdu_iter).get("text"))
                count+=1
            except StopIteration:
                pass

            #15 rom urdu docs
        for _ in range(15):
            try:
                batch.append(next(rom_iter).get("message"))
                count+=1
            except StopIteration:
                pass
                            

            #15 eng docs
        for _ in range(15):
            try:
                batch.append(next(eng_iter).get("text"))
                count+=1
            except StopIteration:
                pass
        if len(batch)>=batch_size:
            yield batch
            batch=[]


    # to get any rem from the 1k docs
    if batch:
        yield batch
                
        

In [9]:
# Train the tokenizer and export
print("Training tokenizer...")
tokenizer.train_from_iterator(batch_iterator(1000,500000),trainer=trainer)


tokenizer.save("tokenizer.json")
print("tokenizer saved to tokenizer.json", "vocab size:", tokenizer.get_vocab_size())

Training tokenizer...


TypeError: 'None' is not an instance of 'str'

In [11]:
# test the tokenizer
text= "یہ ایک ٹیسٹ ہے۔"
tokenizer.encode(text)
print("Encoded text:", tokenizer.encode(text).tokens)
text= "This is a test"
tokenizer.encode(text)
print("Encoded text:", tokenizer.encode(text).tokens)


Encoded text: ['ÛĮÛģ', 'ĠØ§ÛĮÚ©', 'ĠÙ¹ÛĮØ³Ù¹', 'ĠÛģÛĴ', 'ÛĶ']
Encoded text: ['This', 'Ġis', 'Ġa', 'Ġtest']
